# 01 - Load and Prepare Data

This notebook loads CanCM4 decadal prediction datasets (PSL variable),
decodes time coordinates, and stores datasets in a dictionary for further analysis.

Output:
- decadal_datasets dictionary
## Notes
- Data are stored in Google Drive
- NetCDF files use non-standard calendars → handled with cftime
- Output datasets are ready for NAO computation

### import

In [ ]:
from google.colab import drive
import xarray as xr
import cftime
import pandas as pd
import numpy as np
import os

### drive mount

In [ ]:
drive.mount('/content/drive')

### base path and settings

In [ ]:
# Base directory where NetCDF files are stored in Google Drive
base_path = "/content/drive/My Drive/CV/"

# Decadal start years
years = list(range(1960, 2006))

# Ensemble members (r1 to r10)
members = list(range(1, 11))

### loading and decoding function

In [ ]:
# Loop over all decadal start years and ensemble members
# Each dataset corresponds to a 10-year prediction initialized at 'year'
def load_cancm4_psl(base_path, years, members, verbose=True):
    """
    Load CanCM4 decadal PSL datasets from NetCDF files.

    Parameters:
    - base_path: path to folder containing NetCDF files
    - years: list of start years (e.g., 1960–2005)
    - members: list of ensemble members (e.g., 1–10)
    - verbose: print loading progress

    Returns:
    - decadal_datasets: dictionary of xarray datasets
    """

    # Dictionary to store all datasets
    decadal_datasets = {}

    # Loop over each start year and ensemble member
    for year in years:
        for member in members:

            # Build ensemble member string (e.g., r1i1p1)
            member_str = f"r{member}i1p1"

            # Create a unique key name for each dataset
            var_name = f"can_decadal_{year}_r{member}"

            # Construct file name following CanCM4 naming convention
            file_name = f"psl_Amon_CanCM4_decadal{year}_{member_str}_{year+1}01-{year+10}12.nc"

            # Full file path
            file_path = os.path.join(base_path, file_name)

            try:
                # Open dataset without decoding time (avoids calendar issues)
                ds = xr.open_dataset(file_path, decode_times=False)

                # Check if time variable has units (needed for decoding)
                if 'time' in ds and 'units' in ds.time.attrs:

                    # Decode non-standard climate model calendar using cftime
                    decoded = xr.decode_cf(ds, use_cftime=True)

                    # Convert to standard datetime64 for compatibility with pandas/analysis
                    ds['time'] = decoded['time'].astype('datetime64[ns]')

                # Store dataset in dictionary
                decadal_datasets[var_name] = ds

                # Optional progress print
                if verbose:
                    print(f"✔ Loaded: {var_name}")

            except FileNotFoundError:
                # Skip missing files but continue loop
                print(f"Missing file: {file_name}")

            except Exception as e:
                # Catch unexpected errors without stopping execution
                print(f"Error loading {file_name}: {e}")

    return decadal_datasets

### Load datasets

In [ ]:
# Load all CanCM4 PSL datasets into a dictionary
decadal_datasets = load_cancm4_psl(base_path, years, members)

### Quick check

In [ ]:
ds = decadal_datasets["can_decadal_1960_r1"]
print(ds)
print(ds["psl"])
print(ds.time)